# Explainer for Model Predictive Control Maths:

## Discretising Cummins Equation:

Rearrange the cummins equation to isolate the vertical acceleration.


$$
\ddot{\eta}(t)=\left(M + A_\infty\right)^{-1}
\left[
f_{\mathrm{exc}}(t) + f_{\mathrm{PTO}}(t)
- \int_{0}^{t} K(t-\tau)\,\dot{\eta}(\tau)\,d\tau
- C\,\eta(t)
\right]
$$

First we deal with a State Space Realisation of the radiation memory term (convolution integral). We do this for computation efficiency.

We find a small system of equation of order 4-10 that acts as a approximation to the water's memory.

$$\dot{x}_r = A_r x_r(t) + B_r \dot{\eta(t)}$$

$\mu(t) = \int_{0}^{t} K(t-\tau)\,\dot{\eta}(\tau)\,d\tau$

$\mu(t) = C_r x_r(t)$

Here $\dot{x}_r$ are the radiation states and $A_r,B_r,C_r$ can be found using Prony's Method which is described later

Rearranging the cummins equation in terms of $\ddot{\eta}$ with $\mu(t) = C_r x_r(t)$:

$$\ddot{\eta} = (M + A_\infty)^{-1} [(f_{exc} + u(t)) - C \eta - C_r x_r]$$

Let $\ddot{\eta} = \dot{x}$ and write as addition of matrix multiplications:

$$
\dot{x}
=
\underbrace{
\begin{bmatrix}
0 & 1\\
-\dfrac{C}{M_{\mathrm{tot}}} & -\dfrac{C_r}{M_{\mathrm{tot}}}
\end{bmatrix}
}_{A_c}
\begin{bmatrix}
\eta\\
\dot{\eta}
\end{bmatrix}
+
\underbrace{
\begin{bmatrix}
0\\
\dfrac{1}{M_{\mathrm{tot}}}
\end{bmatrix}
}_{B_c}
\left(f_{\mathrm{exc}}+u\right)
$$

$$
\begin{aligned}
\begin{bmatrix}
\dot{\eta}\\
\ddot{\eta}\\
\dot{x}_r
\end{bmatrix}
&=
\underbrace{
\begin{bmatrix}
0 & 1 & \mathbf{0}^{1\times n_r}\\
-\dfrac{C}{M_{\mathrm{tot}}} & 0 & -\dfrac{C_r}{M_{\mathrm{tot}}}\\
\mathbf{0}^{n_r\times 1} & B_r & A_r
\end{bmatrix}
}_{A_c}
\begin{bmatrix}
\eta\\
\dot{\eta}\\
x_r
\end{bmatrix}
+
\underbrace{
\begin{bmatrix}
0\\
\dfrac{1}{M_{\mathrm{tot}}}\\
\mathbf{0}^{n_r\times 1}
\end{bmatrix}
}_{B_c}
\left(f_{\mathrm{exc}}+u\right).
\end{aligned}
$$



Now we want to discretise this system so that we can solve it over a fixed time step $T_{mpc}$

- During the time step (which is very small anyway) we keep the control force constant. This is called a zero-order hold
- First note that the solution to $\dot{x}=A_c x$ is $x(t) = e^{A_c t}x(0)$
- There for to move forward by a discrete amount of time we use $A_d = e^{A_c T_{mpc}}$ where $T_{mpc}$ is the model predictive control timestep that we set
- This is known as a matrix exponential and if $T_{mpc}$ is very small $A_d \approx I + A_cT_{mpc}$ derived by taking the taylor series and omitting 2nd order and greater terms. $I$ is the identity matrix just to confirm.
- To find the discrete version of $B_c$, $B_d$ we integrate the effect of the constant force u over the time step interval. However if $A_c$ is invertible, then it can be written in realtion to $A_c$ and $B_c$.

$$\int_{0}^{T_{mpc}} e^{A_c \tau} B_c d\tau = A_c^{-1}(A_d - I)B_c $$ 


Now for MPC we want a fast way to iterativly solve the equation: To do this we transition to a discrete model. 

- First note that the solution to $\dot{x}=A_c x$ is $x(t) = e^{A_c t}x(0)$
- There for to move forward by a discrete amount of time we use $A_d = e^{A_c T_{mpc}}$ where $T_{mpc}$ is the model predictive control timestep that we set
- This is known as a matrix exponential and if $T_{mpc}$ is very small $A_d \approx I + A_cT_{mpc}$ derived by taking the taylor series and omitting 2nd order and greater terms. $I$ is the identity matrix just to confirm.
- To find the discrete version of $B_c$, $B_d$ we must confront the 


## Prony's Method for approximating Radiation Kernel:

We want to approximate the kernel $K(t)$ with a function $\hat{K(t)}$:

For example we can approximate the kernel as a series of trigonemetrix functions or equally complex exponentials. 

$$\hat{K(t)} = \sum_{i=1}^{n} \alpha_i e^{\beta_i t} $$

Where:
- $n$ is the order of the approximation
- $\alpha_i$ are complex amplitudes
- $\beta_i$ are complex exponents that represent damping and frequency

The Capytaine solver returns the $K_{omega}$ which we can transform into the time domain. 

We then get a time series of $K$ values that we can write as $y_k + a_{1} y_{k-1} + a_{2} y_{k-2} + ... + a_n y_{k-n} = 0$

This basically says that the value of k any time is a weighted sum of the previous points.

We solve for a value using least squares. 

$$y \approx H a $$
$$a = (H^TH)^{-1}H^Ty$$
This result is familiar from Methods of Artificial Intelligence Module.
H is the Hankel matrix which is a matrix of shifted versions of the data. 

We assume that the value $y_k=Cz^k$ then we can rearrange the recurrence relationship to be a polynomial where the roots of the polynomial represent "discrete-time" poles of your system. 

To convert these into continuous-time exponents $\beta$ we use $\beta_i = \frac{\ln{x_i}}{\Delta t}$

Now we have the Beta values we can use least squares to find alpha. 

To cany the result of Prony's Approximation into the radiation matricies we do the following:

- $A_r$ : diagonal elements are $\beta_i$
- $B_r$ : A column vector of ones 
- $C_r$ : A row vector of the amplitudes $\alpha_i$